# Run the complete suite without paid API access

This notebook is a controller and checkpoint viewer. It does **not** claim that a notebook process is a sandbox. Generated code is run only when the capability report confirms Docker isolation; otherwise the notebook stops after inventory/planning and the batch should be dispatched to the GitHub Actions Docker workflow.

Set `SUITE_API_KEY` in the notebook runtime only for the host controller. Never put a literal key in a cell, command argument, artifact, or generated environment.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
ROOT = Path.cwd()
assert (ROOT / 'arena.py').is_file(), 'Run this notebook from the repository root'
print(ROOT)

In [ ]:
# Install only controller dependencies; this does not download model weights.
%pip install -r requirements.txt

In [ ]:
capabilities = subprocess.run([sys.executable, 'tools/notebook_mode.py', '--out', 'capabilities.json'], text=True, capture_output=True, check=True)
print(capabilities.stdout)
caps = json.loads(Path('capabilities.json').read_text())
if caps['execution']['docker_episode'] != 'ready':
    print('Docker isolation is unavailable: inventory and GitHub dispatch are still supported, local generated-code execution is blocked.')

In [ ]:
# Optional host-only provider preflight; it makes no completion request.
if Path('secret_key.json').is_file():
    subprocess.run([sys.executable, 'tools/provider_preflight.py', '--secrets', 'secret_key.json', '--out', 'runs/provider_preflight.json'], check=False)
else:
    print('No secret_key.json: skipping provider preflight; inventory remains fully usable.')

In [ ]:
# `all` expands every declared difficulty-axis combination. Use representative for a cheap pilot.
MATRIX = 'representative'  # change to 'all' for every Cartesian difficulty vector
SEEDS = '0'
subprocess.run([sys.executable, 'tools/suite_inventory.py', '--out', 'suite_manifest.json', '--matrix', MATRIX, '--seeds', SEEDS], check=True)
manifest = json.loads(Path('suite_manifest.json').read_text())
print(manifest['environment_count'], 'environments;', manifest['case_count'], 'planned cases')
assert manifest['registry_audit']['clean'], manifest['issues']

In [ ]:
# Safe preflight: generation and Python compilation only; no provider calls and no Docker containers.
subprocess.run([sys.executable, 'tools/suite_inventory.py', '--out', 'suite_preflight.json', '--matrix', MATRIX, '--seeds', SEEDS, '--preflight'], check=True)

## Run a bounded, resumable Docker batch

The provider, model, and API key environment variable are fixed for the checkpoint. A model failure is recorded and the next case continues; quota/infrastructure errors pause the batch.

In [ ]:
PROVIDER = 'custom'
MODEL = 'replace-with-a-pinned-model-id'
API_BASE = os.environ.get('SUITE_API_BASE', '')
API_KEY_ENV = 'SUITE_API_KEY'
if not os.environ.get(API_KEY_ENV):
    raise RuntimeError('Set the provider key in the host runtime as SUITE_API_KEY; do not paste it into this notebook')
if caps['execution']['docker_episode'] != 'ready':
    raise RuntimeError('No approved isolated executor is available; dispatch this manifest to GitHub Actions instead')
command = [sys.executable, 'tools/run_suite.py', '--manifest', 'suite_manifest.json', '--out', 'runs/notebook-suite', '--provider', PROVIDER, '--model', MODEL, '--api-key-env', API_KEY_ENV, '--sandbox', 'docker', '--max-cases', '1', '--max-api-calls', '100']
if API_BASE:
    command += ['--api-base', API_BASE]
print('Controller command:', ' '.join(command))
subprocess.run(command, check=True)

In [ ]:
# Resume with a larger bound after checking the first artifact.
coverage = json.loads(Path('runs/notebook-suite/coverage.json').read_text())
coverage['counts']